# ÉquiAlgo Challenge - Baseline Model

**Engineering & Computer Science Hackathon 2026 - 24h Challenge**

This notebook loads the data, trains the model currently in production at
our financial partner, evaluates its performance, and conducts an initial fairness audit.

It does not fix anything. That is your job.

Run the entire notebook once (Cell > Run All), read Section 5, and take it from there.

---


## 1. Data Loading

Two files:

| File | Rows | Contains |
|---|---|---|
| `data/donnees_demandes.csv` | 10,000 | historical applications **with** `decision_octroi` |
| `data/candidats_evaluation.csv` | 4,000 | new applications, **without** labels - this is what you will be scored on |


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    selection_rate,
    true_positive_rate,
)

demandes = pd.read_csv('data/donnees_demandes.csv')
candidats = pd.read_csv('data/candidats_evaluation.csv')

print(f'{len(demandes)} historical applications, {len(candidats)} to evaluate')
demandes.head()


: 

In [ ]:
demandes.info()
print()
print(demandes.describe(include='all').T[['count', 'unique', 'top', 'mean']])


## 2. A First Look at the Regions

The committee awarded a scholarship to 40% of applicants. Is this rate the same everywhere?


In [ ]:
# The five administrative regions present in the dataset.
ELOIGNEES = ['Bas-Saint-Laurent', 'Cote-Nord', 'Gaspesie-Iles-de-la-Madeleine']

def groupe_region(df):
    """Groups the five regions into two categories: Major Centers vs. Remote Regions."""
    return np.where(df['region_administrative'].isin(ELOIGNEES), 'Remote', 'Center')

par_region = demandes.groupby('region_administrative').agg(
    applicants=('decision_octroi', 'size'),
    grant_rate=('decision_octroi', 'mean'),
    avg_r_score=('cote_r_equivalent', 'mean'),
).sort_values('grant_rate')
print(par_region.round(3))

demandes['groupe'] = groupe_region(demandes)
print()
print(demandes.groupby('groupe')['decision_octroi'].mean().round(3))


Notice the difference in R score across regions, and then the difference in grant rate.
Is the latter entirely explained by the former? Keep this question open.


## 3. The Production Model

A Random Forest trained on all available columns, including
`region_administrative`. This is the model that the institution currently uses today.


In [ ]:
CATEGORIELLES = ['programme_etudes', 'region_administrative', 'code_postal_3']

def encoder(df_entrainement, df_cible):
    """Encodes categorical columns and aligns columns between the two DataFrames.

    Alignment (reindex) is important: if a category is missing from one of
    the two datasets, get_dummies produces a different number of columns and the model
    fails - or worse, learns on the wrong columns.
    """
    X = pd.get_dummies(
        df_entrainement.drop(columns=['id_candidat', 'decision_octroi', 'groupe'], errors='ignore'),
        columns=CATEGORIELLES,
    )
    Xc = pd.get_dummies(
        df_cible.drop(columns=['id_candidat', 'decision_octroi', 'groupe'], errors='ignore'),
        columns=CATEGORIELLES,
    ).reindex(columns=X.columns, fill_value=0)
    return X, Xc

# Internal train/test split to honestly measure performance.
train, test = train_test_split(
    demandes, test_size=0.3, random_state=42, stratify=demandes['decision_octroi']
)
X_train, X_test = encoder(train, test)
y_train, y_test = train['decision_octroi'], test['decision_octroi']

modele = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
modele.fit(X_train, y_train)
y_pred = modele.predict(X_test)

print(f'Overall Accuracy: {accuracy_score(y_test, y_pred):.2%}')
print()
print(classification_report(y_test, y_pred, digits=3))


About 88% accuracy. From the perspective of a quarterly business dashboard, the model looks great.


## 4. Fairness Audit

Two metrics, and they do not measure the same thing:

- **Demographic Parity** - do both groups receive scholarships at the same rate?
- **Equal Opportunity** - among applicants who deserve the scholarship, do both
  groups have the same chance of obtaining it?

The second metric requires knowing who actually deserves the scholarship. Look closely at what we use to calculate it here.


In [ ]:
groupe_test = groupe_region(test)

cadre = MetricFrame(
    metrics={
        'accuracy': accuracy_score,
        'selection_rate': selection_rate,
        'true_positive_rate': true_positive_rate,
    },
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=groupe_test,
)
print(cadre.by_group.round(4))

ecart_parite = demographic_parity_difference(y_test, y_pred, sensitive_features=groupe_test)
print(f'\nDemographic parity difference: {ecart_parite:.4f}')


The `true_positive_rate` above is computed with respect to `decision_octroi`,
which means with respect to the committee's past decisions.

In other words: we just verified that the model faithfully mimics the committee.
That is not the same as verifying that it is fair.


## 5. The Trap

The immediate instinct is to drop `region_administrative` and call it a day.
Let's measure what actually happens.


In [ ]:
essais = {'full model': []}

def entrainer_sans(colonnes_exclues, etiquette):
    X, Xt = encoder(train, test)
    garder = [c for c in X.columns if not c.startswith(tuple(colonnes_exclues))] if colonnes_exclues else list(X.columns)
    m = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
    m.fit(X[garder], y_train)
    p = m.predict(Xt[garder])
    ecart = demographic_parity_difference(y_test, p, sensitive_features=groupe_test)
    print(f'{etiquette:<42} accuracy {accuracy_score(y_test, p):.4f}   parity gap {ecart:.4f}')
    return ecart

entrainer_sans((), 'All features')
entrainer_sans(('region_administrative_',), 'Without region_administrative')
entrainer_sans(('region_administrative_', 'code_postal_3_'), 'Without region and postal code')


The gap barely changes.

Other columns carry regional information: distance from home to campus, hours
worked, household income, postal code. Removing the label does not remove
the signal. This is the **proxy effect**.

Measure it yourself - a correlation matrix, or a model that tries to predict the
region from other columns, will tell you which ones are the most informative.


## 6. Your Mission

Produce, for the 4,000 applicants in `data/candidats_evaluation.csv`, a grant decision
that is fairer than the current model's.

Three constraints:

1. **Fixed budget.** The institution funds a fixed number of scholarships. Your overall grant rate
   must remain between **36% and 44%**. A submission outside this range receives 0
   on the technical section. Granting scholarships to everyone is not an option.
2. **You are NOT scored on `decision_octroi`.** The jury compares your decisions against a
   reference standard constructed independently of the historical committee, which you do not have.
   Perfectly reproducing `decision_octroi` reproduces the historical bias.
3. **Format.** Two columns, 4,000 rows, values 0 or 1.

The following cell writes a valid submission using the biased baseline model. It serves as a
template. Replace `predictions` with your own model's output.


In [ ]:
X_hist, X_cand = encoder(demandes, candidats)
modele_complet = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
modele_complet.fit(X_hist, demandes['decision_octroi'])
predictions = modele_complet.predict(X_cand)

soumission = pd.DataFrame({
    'id_candidat': candidats['id_candidat'],
    'decision_octroi': predictions.astype(int),
})
soumission.to_csv('predictions.csv', index=False)

taux = soumission['decision_octroi'].mean()
print(f'{len(soumission)} rows written to predictions.csv')
print(f'Grant rate: {taux:.1%}', '- OK' if 0.36 <= taux <= 0.44 else '- OUT OF BUDGET')
soumission.head()


---

### Next Steps & Exploration Ideas

- Which columns best predict the region? Those are your proxy variables.
- Between Demographic Parity and Equal Opportunity, which one would you defend before an
  ethics committee, and why? Both cannot be satisfied simultaneously
  when the two groups do not have identical feature distributions.
- `fairlearn.postprocessing.ThresholdOptimizer` and `fairlearn.reductions.ExponentiatedGradient`
  are the two fastest tools to set up. The first adjusts decision thresholds post-hoc,
  while the second retrains the model under fairness constraints.
- Sweep the fairness constraint parameter and plot the trade-off curve (fairness vs. utility):
  this is your Pareto front, and the jury will ask you to justify your chosen operating point.
- If `decision_octroi` is a biased label, is the best model really the one that predicts it most accurately?
